# Screening Visualization and Count Check Notebook for Synthetic CSV Data

This notebook is for **exploratory checks** (no training is performed).

1. Load CTMC CSV files in the specified directory and verify record counts before/after screening
2. Visualize the distribution of `λ_i = Q'[i, i+1]` extracted from `Q' (= Q_mle)` and inspect whether the screening behavior is reasonable

- **Input**: `data_dir` (directory containing CSV files)
- **Output**: kept/dropped counts, drop-reason summary, `λ` visualization (supports mixed N)


In [ ]:
# ==== Edit these first ====
# data_dir: directory containing CSV files
# recursive: whether to search subdirectories

data_dir = "/mnt/ssd/datas/discrete_train_with_mle/"  # <- edit first
recursive = True

# Screening thresholds
min_lambda = 1e-8
max_lambda = 1e6

# Toggle screening options
check_nan_inf = True
require_structure = True
max_abs_diag_diff = 1e-8

# Display settings
top_k_dropped_examples = 10
# Maximum number of entries to fully parse for visualization (for kept/dropped each)
max_visualize_per_group = 10000


In [ ]:
from __future__ import annotations

import sys
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import japanize_matplotlib

import numpy as np

# Make imports robust regardless of notebook execution location
repo_root = Path.cwd()
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from ctmc_surrogate.data.dataset_csv_loader import parse_ctmc_csv
from ctmc_surrogate.data.dataset_screening import (
    ScreeningConfig,
    extract_lambdas_from_Q,
    screen_dir_fast,
)


In [ ]:
# Run fast screening first without loading samples
cfg = ScreeningConfig(
    min_lambda=min_lambda,
    max_lambda=max_lambda,
    check_nan_inf=check_nan_inf,
    require_structure=require_structure,
    max_abs_diag_diff=max_abs_diag_diff,
)

fast = screen_dir_fast(data_dir, recursive=recursive, cfg=cfg)

print(f"Total files: {fast['total']}")
print(f"Kept files: {len(fast['kept_paths'])}")
print(f"Dropped files: {len(fast['dropped_paths'])}")

print("\n[Dropped reason counts]")
if fast['drop_counts']:
    for reason, cnt in sorted(fast['drop_counts'].items(), key=lambda x: x[1], reverse=True):
        print(f"- {reason}: {cnt}")
else:
    print("- No dropped files")

print(f"\n[Top {top_k_dropped_examples} dropped examples]")
for i, item in enumerate(fast['report'][:top_k_dropped_examples], start=1):
    path = item.get("path", "(no path)")
    reason = item.get("reason", "(no reason)")
    idx = item.get("index")
    lam = item.get("lambda")
    detail = item.get("detail")
    base = f"{i:02d}. path={path}, reason={reason}"
    if idx is not None and lam is not None:
        base += f", index={idx}, lambda={lam:.6e}"
    if detail:
        base += f", detail={detail}"
    print(base)
